In [4]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import numpy as np

# 加载 MNIST 数据集
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# 数据预处理
x_train = x_train.reshape(-1, 784).astype('float32') / 255.0
x_test = x_test.reshape(-1, 784).astype('float32') / 255.0
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# 创建数据生成器
def next_batch(data_x, data_y, batch_size):
    indices = np.random.choice(len(data_x), batch_size, replace=False)
    return data_x[indices], data_y[indices]

# 其余参数保持不变
learning_rate = 1e-4
keep_prob_rate = 0.7 # dropout 0.3
max_epoch = 2000
batch_size = 100

# 启用 TensorFlow 1.x 兼容模式
tf.compat.v1.disable_eager_execution()
# 重置计算图
tf.compat.v1.reset_default_graph()

# 定义计算图
with tf.compat.v1.Session() as sess:
    # 定义占位符
    xs = tf.compat.v1.placeholder(tf.float32, [None, 784])
    ys = tf.compat.v1.placeholder(tf.float32, [None, 10])
    keep_prob = tf.compat.v1.placeholder(tf.float32)
    
    # 网络结构定义（与原始代码相同）
    def weight_variable(shape):
        initial = tf.compat.v1.truncated_normal(shape, stddev=0.1)
        return tf.Variable(initial)
    
    def bias_variable(shape):
        initial = tf.constant(0.1, shape=shape)
        return tf.Variable(initial)
    
    def conv2d(x, W):
        return tf.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')
    
    def max_pool_2x2(x):
        return tf.nn.max_pool2d(x, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME')
    
    x_image = tf.reshape(xs, [-1, 28, 28, 1])
    
    # 卷积层 1
    W_conv1 = weight_variable([7, 7, 1, 32])
    b_conv1 = bias_variable([32])
    h_conv1 = tf.nn.relu(conv2d(x_image, W_conv1) + b_conv1)
    h_pool1 = max_pool_2x2(h_conv1)
    
    # 卷积层 2
    W_conv2 = weight_variable([5, 5, 32, 64])
    b_conv2 = bias_variable([64])
    h_conv2 = tf.nn.relu(conv2d(h_pool1, W_conv2) + b_conv2)
    h_pool2 = max_pool_2x2(h_conv2)
    
    # 全连接层
    W_fc1 = weight_variable([7 * 7 * 64, 1024])
    b_fc1 = bias_variable([1024])
    h_pool2_flat = tf.reshape(h_pool2, [-1, 7 * 7 * 64])
    h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, W_fc1) + b_fc1)
    # h_fc1_drop = tf.nn.dropout(h_fc1, keep_prob)
     # 移除dropout
    h_fc1_drop = h_fc1
    
    # 输出层
    W_fc2 = weight_variable([1024, 10])
    b_fc2 = bias_variable([10])
    prediction = tf.nn.softmax(tf.matmul(h_fc1_drop, W_fc2) + b_fc2)
    
    # 损失函数和优化器 修复：将 reduction_indices 改为 axis
    cross_entropy = tf.reduce_mean(-tf.reduce_sum(ys * tf.math.log(prediction + 1e-10), axis=[1]))
    train_step = tf.compat.v1.train.AdamOptimizer(learning_rate).minimize(cross_entropy)
    
    # 准确率计算
    correct_prediction = tf.equal(tf.argmax(prediction, 1), tf.argmax(ys, 1))
    accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
    
    # 初始化变量
    init = tf.compat.v1.global_variables_initializer()
    sess.run(init)
    
    # 训练
    for i in range(max_epoch):
        batch_x, batch_y = next_batch(x_train, y_train, batch_size)
        sess.run(train_step, feed_dict={xs: batch_x, ys: batch_y, keep_prob: keep_prob_rate})
        
        if i % 100 == 0:
            train_acc = sess.run(accuracy, feed_dict={xs: batch_x, ys: batch_y, keep_prob: 1.0})
            test_acc = sess.run(accuracy, feed_dict={xs: x_test[:1000], ys: y_test[:1000], keep_prob: 1.0})
            print(f'Step {i}, Training accuracy: {train_acc:.4f}, Test accuracy: {test_acc:.4f}')

    
     # 最终测试准确率
    final_acc = sess.run(accuracy, feed_dict={xs: x_test, ys: y_test, keep_prob: 1.0})
    print(f'Final test accuracy: {final_acc:.4f}')

Step 0, Training accuracy: 0.1200, Test accuracy: 0.1150
Step 100, Training accuracy: 0.0800, Test accuracy: 0.1260
Step 200, Training accuracy: 0.1600, Test accuracy: 0.1260
Step 300, Training accuracy: 0.1300, Test accuracy: 0.1260
Step 400, Training accuracy: 0.0900, Test accuracy: 0.1260
Step 500, Training accuracy: 0.1000, Test accuracy: 0.1260
Step 600, Training accuracy: 0.1000, Test accuracy: 0.1260
Step 700, Training accuracy: 0.1400, Test accuracy: 0.1260
Step 800, Training accuracy: 0.1100, Test accuracy: 0.1260
Step 900, Training accuracy: 0.0900, Test accuracy: 0.1260
Step 1000, Training accuracy: 0.1300, Test accuracy: 0.1260
Step 1100, Training accuracy: 0.1000, Test accuracy: 0.1260
Step 1200, Training accuracy: 0.1700, Test accuracy: 0.1260
Step 1300, Training accuracy: 0.0800, Test accuracy: 0.1260
Step 1400, Training accuracy: 0.0600, Test accuracy: 0.1260
Step 1500, Training accuracy: 0.1100, Test accuracy: 0.1260
Step 1600, Training accuracy: 0.1000, Test accuracy:

- 可以看到，上面的效果几乎随机，可能是训错了方向
- 说明原版本的代码框架给的henbuhao
- 我们还是手动实现一下
- 请看下方

In [21]:
import tensorflow as tf
import numpy as np

# 数据预处理
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()
train_images = tf.cast(train_images[..., tf.newaxis]/255.0, tf.float32)  # 添加通道维度并归一化
test_images = tf.cast(test_images[..., tf.newaxis]/255.0, tf.float32)
train_labels = tf.one_hot(train_labels, 10)  # 手动one-hot编码
test_labels = tf.one_hot(test_labels, 10)

# 初始化参数
def weight_variable(shape):
    return tf.Variable(tf.random.truncated_normal(shape, stddev=0.1))

def bias_variable(shape):
    return tf.Variable(tf.constant(0.1, shape=shape))

# 卷积和池化操作
def conv2d(x, W):
    return tf.nn.conv2d(x, W, strides=[1,1,1,1], padding='SAME')

def max_pool_2x2(x):
    return tf.nn.max_pool2d(x, ksize=2, strides=2, padding='SAME')

# 构建网络
class ManualCNN(tf.Module):
    def __init__(self):
        super().__init__()
        # 卷积层1 - 7x7的大，能看清楚每一个细节
        self.W_conv1 = weight_variable([7,7,1,32])
        self.b_conv1 = bias_variable([32])
        
        # 卷积层2 - 5x5的小，专注局部特征
        self.W_conv2 = weight_variable([5,5,32,64])
        self.b_conv2 = bias_variable([64])
        
        # 全连接层 
        self.W_fc1 = weight_variable([7 * 7 * 64, 1024])
        self.b_fc1 = bias_variable([1024])
        self.W_fc2 = weight_variable([1024,10])
        self.b_fc2 = bias_variable([10])
    
    def __call__(self, x, training=False):
        # 输入重塑 - 给数据整容，变成适合卷积的形状
        x = tf.reshape(x, [-1,28,28,1])
        
        # 第一眼：大范围扫描
        h_conv1 = tf.nn.relu(conv2d(x, self.W_conv1) + self.b_conv1)
        h_pool1 = max_pool_2x2(h_conv1)  # 池化：只记住重点
        
        # 第二眼：细节观察
        h_conv2 = tf.nn.relu(conv2d(h_pool1, self.W_conv2) + self.b_conv2)
        h_pool2 = max_pool_2x2(h_conv2)  # 再次池化：提炼精华
        
        
        h_pool2_flat = tf.reshape(h_pool2, [-1,7 * 7 * 64])
        h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, self.W_fc1) + self.b_fc1)
        
        # 训练时应用Dropout - 防止过拟合
        if training:
            h_fc1 = tf.nn.dropout(h_fc1, rate=0.3)
        
        # 最终决策：这是几？
        return tf.nn.softmax(tf.matmul(h_fc1, self.W_fc2) + self.b_fc2)

# CNN模型
model = ManualCNN()
# Adam优化器，学习率1e-4
optimizer = tf.optimizers.Adam(1e-4)

# 手动实现训练循环 - 让模型学会认
def train_step(images, labels):
    with tf.GradientTape() as tape:
        predictions = model(images, training=True)
        # 计算交叉熵损失 - 看看模型猜得有多离谱
        loss = -tf.reduce_mean(tf.reduce_sum(labels * tf.math.log(predictions+1e-10), axis=1))
    # 反向传播：找出错误，修正模型
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

# 计算准确率 - 给模型考试打分
def compute_accuracy(images, labels):
    predictions = model(images)
    correct = tf.equal(tf.argmax(predictions,1), tf.argmax(labels,1))
    return tf.reduce_mean(tf.cast(correct, tf.float32))

# 训练开始！看看模型能考多少分
for epoch in range(2000):
    # 随机抽取100个样本组成小批量
    idx = np.random.choice(train_images.shape[0], 100) #len(train_images)
    batch_images = tf.gather(train_images, idx)
    batch_labels = tf.gather(train_labels, idx)
    
    # 一次训练步骤
    loss = train_step(batch_images, batch_labels)
    
    # 每100轮考一次试，看看进步如何
    if epoch % 100 == 0:
        acc = compute_accuracy(test_images[:1000], test_labels[:1000])
        print("Epoch {}: 损失={}, 准确率={}".format(epoch, loss, acc))
        # 如果准确率超过0.9，可以小小庆祝一下！
        if acc > 0.9:
            print("  ✨ 不错哦，准确率超过90%了！")

Epoch 0: 损失=Tensor("Neg_2003:0", shape=(), dtype=float32), 准确率=Tensor("Mean_2029:0", shape=(), dtype=float32)


OperatorNotAllowedInGraphError: Using a symbolic `tf.Tensor` as a Python `bool` is not allowed. You can attempt the following resolutions to the problem: If you are running in Graph mode, use Eager execution mode or decorate this function with @tf.function. If you are using AutoGraph, you can try decorating this function with @tf.function. If that does not work, then you may be using an unsupported feature or your source code may not be visible to AutoGraph. See https://github.com/tensorflow/tensorflow/blob/master/tensorflow/python/autograph/g3doc/reference/limitations.md#access-to-source-code for more information.

In [20]:
batch_images, batch_labels, acc

(<tf.Tensor 'GatherV2_4004:0' shape=(100, 28, 28, 1) dtype=float32>,
 <tf.Tensor 'GatherV2_4005:0' shape=(100, 10) dtype=float32>,
 <tf.Tensor 'Mean_2027:0' shape=() dtype=float32>)

In [ ]:
import tensorflow as tf
import numpy as np

# 手动实现数据预处理
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()
train_images = tf.cast(train_images[..., tf.newaxis]/255.0, tf.float32)  # 添加通道维度并归一化
test_images = tf.cast(test_images[..., tf.newaxis]/255.0, tf.float32)
train_labels = tf.one_hot(train_labels, 10)  # 手动one-hot编码
test_labels = tf.one_hot(test_labels, 10)

# 初始化参数
def weight_variable(shape):
    return tf.Variable(tf.random.truncated_normal(shape, stddev=0.1))

def bias_variable(shape):
    return tf.Variable(tf.constant(0.1, shape=shape))

# 卷积和池化操作
def conv2d(x, W):
    return tf.nn.conv2d(x, W, strides=[1,1,1,1], padding='SAME')

def max_pool_2x2(x):
    return tf.nn.max_pool2d(x, ksize=2, strides=2, padding='SAME')

# 构建网络
class ManualCNN(tf.Module):
    def __init__(self):
        super().__init__()
        # 卷积层1
        self.W_conv1 = weight_variable([7,7,1,32])
        self.b_conv1 = bias_variable([32])
        
        # 卷积层2
        self.W_conv2 = weight_variable([5,5,32,64])
        self.b_conv2 = bias_variable([64])
        
        # 全连接层
        self.W_fc1 = weight_variable([7 * 7 * 64, 1024])
        self.b_fc1 = bias_variable([1024])
        self.W_fc2 = weight_variable([1024,10])
        self.b_fc2 = bias_variable([10])
    
    def __call__(self, x, training=False):
        # 输入重塑
        x = tf.reshape(x, [-1,28,28,1])
        
        # 卷积层1 + ReLU + 池化
        h_conv1 = tf.nn.relu(conv2d(x, self.W_conv1) + self.b_conv1)
        h_pool1 = max_pool_2x2(h_conv1)
        
        # 卷积层2 + ReLU + 池化
        h_conv2 = tf.nn.relu(conv2d(h_pool1, self.W_conv2) + self.b_conv2)
        h_pool2 = max_pool_2x2(h_conv2)
        
        # 全连接层
        h_pool2_flat = tf.reshape(h_pool2, [-1,7 * 7 * 64])
        h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, self.W_fc1) + self.b_fc1)
        
        # 训练时应用Dropout
        if training:
            h_fc1 = tf.nn.dropout(h_fc1, rate=0.3)
        
        return tf.nn.softmax(tf.matmul(h_fc1, self.W_fc2) + self.b_fc2)

# 实例化模型和优化器
model = ManualCNN()
optimizer = tf.optimizers.Adam(1e-4)

# 手动实现训练循环
def train_step(images, labels):
    with tf.GradientTape() as tape:
        predictions = model(images, training=True)
        # 手动计算交叉熵损失
        loss = -tf.reduce_mean(tf.reduce_sum(labels * tf.math.log(predictions), axis=1))
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

# 计算准确率
def compute_accuracy(images, labels):
    predictions = model(images)
    correct = tf.equal(tf.argmax(predictions,1), tf.argmax(labels,1))
    return tf.reduce_mean(tf.cast(correct, tf.float32))

# 训练过程
for epoch in range(2000):
    # 生成batch
    idx = np.random.choice(train_images.shape[0], 100)
    batch_images = tf.gather(train_images, idx)
    batch_labels = tf.gather(train_labels, idx)
    
    loss = train_step(batch_images, batch_labels)
    
    if epoch % 100 == 0:
        acc = compute_accuracy(test_images[:1000], test_labels[:1000])
        print(f"Epoch {epoch}: Loss={loss:.4f}, Accuracy={acc:.4f}")

TypeError: len is not well defined for a symbolic Tensor (Cast_2045:0). Please call `x.shape` rather than `len(x)` for shape information.